# [CORRECCIÓN TRIBUNAL] Laplaciano Hermítico Complejo con Skip-Connections
El código original usaba una matriz asimétrica real. A continuación se provee la implementación oficial validada por el tribunal.

In [ ]:
import numpy as np
import math
import torch

def build_magnetic_laplacian_polydim(N, q, skip_k=2):
    """
    Laplaciano Magnético para POLYDIM V3.2
    Basado en Fanuel et al. (2017) y Furutani et al. (2020)
    """
    # 1. Matriz de adyacencia dirigida con skip-connections
    A = np.zeros((N, N), dtype=np.float64)
    for i in range(N - 1):
        A[i, i + 1] = 1.0  # Cadena base (forward)
    for k in range(2, skip_k + 1):  # Skip connections
        for i in range(N - k):
            A[i, i + k] = 1.0

    # 2. Matriz simetrizada y grado
    A_s = 0.5 * (A + A.T)
    D_s = np.diag(np.sum(A_s, axis=1))

    # 3. Fase magnética: Theta_ij = 2*pi*q * (A_ij - A_ji)
    Theta = 2 * np.pi * q * (A - A.T)

    # 4. Matriz de Hermite con fase compleja
    H_q = A_s * np.exp(1j * Theta)

    # 5. Laplaciano Magnético: L^(q) = D_s - H^(q)
    L_q = D_s - H_q

    return L_q, A, A_s

# Test the function
N = 10
q = 0.25
L_q, A, A_s = build_magnetic_laplacian_polydim(N, q, skip_k=2)
L_q_tensor = torch.tensor(L_q, dtype=torch.cfloat)
print("Laplaciano Hermítico Complejo construido con éxito.")



# POLYDIM V3.2: Matemática Topológica Core

Este cuaderno demuestra la viabilidad matemática de las dos innovaciones clave que diferencian a POLYDIM de los Transformers estándar: el **Laplaciano Magnético Causal** (que evita la fuga de información) y las **Rotaciones Isométricas** para representaciones latentes distribuidas.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

print("Bibliotecas cargadas.")


## 1. El Laplaciano Magnético $L^{(q)}$
La fuga de información (data leakage) en atención simétrica se rompe introduciendo una fase magnética $q$.


In [ ]:
N = 10
q = math.pi / 4.0

H_q = np.zeros((N, N))
deg = np.zeros(N)

for i in range(1, N):
    j = i - 1
    H_q[j, i] = math.cos(q)
    H_q[i, j] = math.cos(2*q)
    deg[i] += 1
    deg[j] += 1

L0 = np.diag(deg) - H_q

plt.imshow(L0, cmap="coolwarm")
plt.title("Laplaciano Magnético (q=pi/4)")
plt.colorbar()
plt.show()

print("Notar la asimetría: El flujo hacia adelante tiene más peso que el flujo hacia atrás, introduciendo causalidad estricta.")


## 2. Inyectividad del ECP (Hash)
En la versión V3.2, usamos hashes SHA-256 de las matrices de rotación para evitar colisiones semánticas (anagramas).


In [ ]:
import hashlib

def generate_ecp_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()[:8]

print(f"ACT -> {generate_ecp_hash('ACT')}")
print(f"CAT -> {generate_ecp_hash('CAT')}")
print("¡Cero colisiones!")
